In [ ]:
import os
import utils
import random
import numpy as np
import pickle as pkl
from sklearn.metrics import roc_auc_score
from collections import defaultdict

from tqdm import tqdm
from torch.optim import Adam
from sklearn.model_selection import StratifiedKFold

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Configuration

In [ ]:
_DATA = 'diabetes'
_NUM_CONDITION = 20
_NUM_CONDITION_BATCH = 10
_SEED = 42
_SHOT = 0
_NUM_EXAMPLE = 5
_NUM_TRIAL = 5

llm_model = 'gpt-4o'
#_API_KEY = ''

# Train/Test Split

In [ ]:
utils.set_seed(_SEED)
df, X_train, X_test, y_train, y_test, target_attr, label_list, is_cat = utils.get_dataset(_DATA, _SEED, _SHOT)
multiclass = True if len(label_list) > 2 else False
y_test = [label_list.index(l) for l in y_test]

X_all = df.drop(target_attr, axis=1)
y_all = np.array([label_list.index(l) for l in df[target_attr]])

In [ ]:
if _DATA in ['communities', 'myocardial']:
    with open(f'data/{_DATA}_selected_column_indices.pkl', 'rb') as f:
        selected_column_indices = pkl.load(f)
    
    X_all = X_all.iloc[:,selected_column_indices]
    X_test = X_test.iloc[:,selected_column_indices]
    
    if X_train is not None:
        X_train = X_train.iloc[:,selected_column_indices]

X_all

# Data Information

In [ ]:
TASK_DICT = {
    'blood': "Did the person donate blood? Yes or no?",
    'credit-g': "Does this person receive a credit? Yes or no?",
    'diabetes': "Does this patient have diabetes? Yes or no?",
    'heart': "Does the coronary angiography of this patient show a heart disease? Yes or no?",
    'adult': "Does this person earn more than 50000 dollars per year? Yes or no?",
    'bank': "Does this client subscribe to a term deposit? Yes or no?",
    'car': "How would you rate the decision to buy this car? Unacceptable, acceptable, good or very good?",
    'communities': "How high will the rate of violent crimes per 100K population be in this area. Low, medium, or high?",
    'myocardial': "Does the myocardial infarction complications data of this patient show chronic heart failure? Yes or no?"
}

task_desc = TASK_DICT[_DATA]
print(task_desc)

In [ ]:
meta_data_name = f"./data/{_DATA}-metadata.json"
feature_desc = utils.get_feature_desc(X_all, is_cat, meta_data_name)
print(feature_desc)

In [ ]:
label_desc = "[" + ','.join([str(l) for l in label_list]) + "]"
print(label_desc)

# Generate Conditions

In [ ]:
# Zero-shot
template_generate_conditions = 'templates/generate_conditions.txt'

for trial in range(_NUM_TRIAL):
    random.seed(42 + trial)

    label_conditions_all = [[] for _ in range(len(label_list))]
    label_descriptions_all = [[] for _ in range(len(label_list))]
    condition2index_all = defaultdict(list)

    while sum([len(condition2index_all[label_index]) >= _NUM_CONDITION for label_index in range(len(label_list))]) < len(label_list):
    
        ##### 1. Write the Prompt #####
        example_indices = random.sample(list(X_all.index), k=_NUM_EXAMPLE)
        
        examples = ''
        for i, _index in enumerate(example_indices):
            examples += f'Sample {i+1}:\n'
            examples += X_all.iloc[int(_index)].to_string() + '\n\n'
        
        fill_in_dict = {
            "[NUM_CONDITION]": str(_NUM_CONDITION_BATCH),
            "[TASK]": task_desc,
            "[FEATURES]": feature_desc,
            "[EXAMPLES]": examples,
            "[LABEL_LIST]": label_desc
        }
    
        ##### 2. Generate Groups #####
        prompt = utils.get_prompt(template_generate_conditions, fill_in_dict)
        response = utils.query_gpt(prompt, _API_KEY, max_tokens=2500, temperature=0, model=llm_model)

        response = response.replace('\n\n', '\n')
    
        label_conditions, label_descriptions = utils.parse_conditions(response, label_list, "Label")
    
        ##### 3. Collect Samples for Each Group #####
        condition2index = defaultdict(list)
        
        for label_index, conditions in enumerate(label_conditions):
            for condition_index, condition in tqdm(enumerate(conditions), total=len(conditions)):
        
                # 3.1. Generate the function
                template_ask_for_function = 'templates/ask_for_function.txt'
        
                fill_in_dict = {
                    "[NAME]": f"condition_{label_index}_{condition_index}",
                    "[FEATURES]": feature_desc,
                    "[CONDITIONS]": condition
                }
        
                prompt = utils.get_prompt(template_ask_for_function, fill_in_dict)
                response = utils.query_gpt(prompt, _API_KEY, max_tokens=1500, temperature=0, model='gpt-4o-mini')
        
                # 3.2. Run the function
                function_str = response.split('<start>')[1].split('<end>')[0].strip()
                index_list = utils.run_function(function_str, X_all)
    
                if 'Error 1' in index_list or 'Error 2' in index_list:
                    print('Error', condition)
                    continue

                condition2index[label_index].append(sorted(index_list))

        for label_index in range(len(label_list)):
            for i, new_indices in enumerate(condition2index[label_index]):
                if (not sum([new_indices == indices for indices in condition2index_all[label_index]])) and (len(new_indices) > 0):
                    condition2index_all[label_index].append(new_indices)
                    label_conditions_all[label_index].append(label_conditions[label_index][i])
                    label_descriptions_all[label_index].append(label_descriptions[label_index][i])

        print([len(condition2index_all[label_index]) for label_index in range(len(label_list))])

    for label_index in range(len(label_list)):
        condition2index_all[label_index] = condition2index_all[label_index][:_NUM_CONDITION]
        label_conditions_all[label_index] = label_conditions_all[label_index][:_NUM_CONDITION]
        label_descriptions_all[label_index] = label_descriptions_all[label_index][:_NUM_CONDITION]
    
    ##### 4. Print Results #####
    print(f'TRIAL {trial+1}')
    for label_index, conditions in enumerate(label_conditions_all):
        label_name = label_list[label_index]
        print(f'\n=== Label: {label_name} ===')
        print(f'{"Idx":<5}{"Condition":<60}{"#Samples":>12}')
        print('-' * 80)
    
        for condition_index, condition in enumerate(conditions):
            condition_size = len(condition2index_all[label_index][condition_index])
            print(f'{condition_index+1:<5}{condition:<60}{condition_size:>6}/{X_all.shape[0]:<6}')
    
    print()
    
    ##### 5. Save Results #####
    save_dir = f'conditions/{_SHOT}_shot/{trial}_trial/{llm_model}'
    os.makedirs(save_dir, exist_ok=True)

    with open(f'{save_dir}/{_DATA}_{_NUM_CONDITION}_{_SEED}_{_NUM_EXAMPLE}_conditions.pkl', 'wb') as f:
        pkl.dump([label_conditions_all, label_descriptions_all], f)
    
    with open(f'{save_dir}/{_DATA}_{_NUM_CONDITION}_{_SEED}_{_NUM_EXAMPLE}_condition2index.pkl', 'wb') as f:
        pkl.dump(condition2index_all, f)

# Evaluation

In [ ]:
_NUM_CONDITION = 20
_NUM_EXAMPLE = 5

In [ ]:
normalize2auroc = []

for trial in range(_NUM_TRIAL):
    save_dir = f'conditions/{_SHOT}_shot/{trial}_trial/{llm_model}'

    with open(f'{save_dir}/{_DATA}_{_NUM_CONDITION}_{_SEED}_{_NUM_EXAMPLE}_conditions.pkl', 'rb') as f:
        label_conditions, label_descriptions = pkl.load(f)
    
    with open(f'{save_dir}/{_DATA}_{_NUM_CONDITION}_{_SEED}_{_NUM_EXAMPLE}_condition2index.pkl', 'rb') as f:
        condition2index = pkl.load(f)
    
    X = utils.generate_feature(condition2index, X_all.shape[0], len(label_list), _NUM_CONDITION)
    auroc = utils.evaluate(X, X_test.index, y_all, multiclass, len(label_list), _NUM_CONDITION)
    normalize2auroc.append(auroc)

In [ ]:
print(f'{"Mean":>7} | {"Std":>7} | {"Min":>7} | {"Max":>7}')
print('-' * 50)

vals = normalize2auroc
print(vals)
auroc_mean = np.mean(vals)
auroc_std = np.std(vals)
auroc_min = np.min(vals)
auroc_max = np.max(vals)

print(f'{auroc_mean:7.4f} | {auroc_std:7.4f} | {auroc_min:7.4f} | {auroc_max:7.4f}')